# 步骤四：大语言模型（LLM）评估

使用 Ollama 对 AI 生成代码进行代码审查，与 lab4 人类 PR 结果对比。

**两个任务：**
- Task 1: Merge Prediction（是否应该合并）
- Task 2: Review Comment Generation（代码审查评论）

**4 种 Prompt 类型：**
- zero_shot / few_shot / cot / role_based

**4 种上下文类型：**
- diff_only / diff_pr_desc / diff_commit / diff_extra

In [1]:
import sys
sys.path.insert(0, '..')
sys.path.append('../../lab4')

import importlib
import config
importlib.reload(config)

<module 'config' from 'f:\\学习资料\\智能软件工程实践\\lab5\\step4_llm\\..\\config.py'>

## 1. 数据准备

加载 step1 筛选出的 AI PR，提取字段，构建 4 种上下文。

In [2]:
from data_prepare import AIPRDataPreparer

preparer = AIPRDataPreparer()
contexts, summary = preparer.run_all()

print(f"\n数据摘要:")
print(f"  PR 总数: {summary['total_prs']}")
print(f"  Merged: {summary['merged']}")
print(f"  Not Merged: {summary['not_merged']}")
print(f"  平均 diff 长度: {summary['avg_diff_length']:.0f} 字符")

步骤四数据准备: 加载 AI PR → 提取字段 → 构建上下文

[1/4] 加载 AI PR 数据...
  加载 kubernetes_kubernetes: 5 个 AI PR
  加载 pytorch_pytorch: 84 个 AI PR
  加载 tensorflow_tensorflow: 3 个 AI PR
  加载 microsoft_vscode: 183 个 AI PR
  加载 langchain-ai_langchain: 69 个 AI PR
  共加载 344 个 AI PR

[2/4] 随机采样 100 条 PR...
  采样完成，共 100 条

[3/4] 提取字段...
  提取完成，共 100 条

[4/4] 构建 4 种上下文...
  已处理 50/100 条...
  已处理 100/100 条...
  上下文构建完成，共 100 条

摘要: 100 个 PR, merged=41, not_merged=59

数据摘要:
  PR 总数: 100
  Merged: 41
  Not Merged: 59
  平均 diff 长度: 166220 字符


## 2. LLM 推理 + 评估

调用 Ollama 对每条 AI PR 进行推理，然后评估指标。

**注意：需要先启动 Ollama 服务** `ollama serve`

In [3]:
from evaluate import LLMEvaluator

evaluator = LLMEvaluator()
results = evaluator.run_all()

evaluator.print_summary()

步骤四: LLM 代码审查测试 (Ollama)
加载上下文缓存: f:\学习资料\智能软件工程实践\lab5\data\step4_contexts.json
  已加载 100 条上下文
已加载已有结果: 850 条记录

组 1/8: merge_prediction + zero_shot
  文件: f:\学习资料\智能软件工程实践\lab5\results\step4\merge_prediction_zero_shot.json


[merge_prediction][zero_shot]: 100%|███████████| 400/400 [00:01<00:00, 368.61req/s, new=0, skip=400]


  跳过 400 条已有记录
  加载已有 400 条，新增 0 条
  本组完成: 400 条记录

  预测分布: Yes=321, No=78, Unknown=1
  Accuracy : 0.5375
  Precision: 0.4673
  Recall   : 0.9146
  F1-score : 0.6186
  ROC-AUC  : 0.5948

组 2/8: merge_prediction + few_shot
  文件: f:\学习资料\智能软件工程实践\lab5\results\step4\merge_prediction_few_shot.json


[merge_prediction][few_shot]: 100%|████████████| 400/400 [00:00<00:00, 441.89req/s, new=0, skip=400]


  跳过 400 条已有记录
  加载已有 400 条，新增 0 条
  本组完成: 400 条记录

  预测分布: Yes=314, No=86, Unknown=0
  Accuracy : 0.4900
  Precision: 0.4363
  Recall   : 0.8354
  F1-score : 0.5732
  ROC-AUC  : 0.5427

组 3/8: merge_prediction + cot
  文件: f:\学习资料\智能软件工程实践\lab5\results\step4\merge_prediction_cot.json


...

  本组完成: 400 条记录

组 7/8: review_comment + cot
  文件: f:\学习资料\智能软件工程实践\lab5\results\step4\review_comment_cot.json


[review_comment][cot]: 100%|███████| 400/400 [1:42:51<00:00, 15.43s/req, new=400, lat=17.1s, skip=0]


  本组完成: 400 条记录

组 8/8: review_comment + role_based
  文件: f:\学习资料\智能软件工程实践\lab5\results\step4\review_comment_role_based.json


[review_comment][role_based]: 100%|█| 400/400 [1:29:20<00:00, 13.40s/req, new=400, lat=19.2s, skip=0


  本组完成: 400 条记录

全局评估

Merge Prediction 结果汇总

--- zero_shot ---
  Context          Acc      Prec     Rec      F1       AUC      N     
  ------------------------------------------------------------
  diff_only        0.5000   0.4483   0.9512   0.6094   0.5684   200   
  diff_pr_desc     0.4400   0.4227   1.0000   0.5942   0.5254   200   
  diff_commit      0.4500   0.4271   1.0000   0.5985   0.5339   200   
  diff_extra       0.7600   0.7073   0.7073   0.7073   0.7520   200   

--- few_shot ---
  Context          Acc      Prec     Rec      F1       AUC      N     
  ------------------------------------------------------------
  diff_only        0.4900   0.4432   0.9512   0.6047   0.5604   200   
  diff_pr_desc     0.4400   0.4157   0.9024   0.5692   0.5105   200   
  diff_commit      0.4400   0.4157   0.9024   0.5692   0.5105   200   
  diff_extra       0.5900   0.5000   0.5854   0.5393   0.5893   200   

--- cot ---
  Context          Acc      Prec     Rec      F1       AUC      N    

## 3. 断点续传

中断后重新运行 `evaluator.run_all()` 即可自动跳过已完成记录，从断点继续。

结果按 `(task, prompt_type)` 保存到独立文件：
- `lab5/results/step4/merge_prediction_zero_shot.json`
- `lab5/results/step4/merge_prediction_few_shot.json`
- ... 共 8 个文件